# The story so far

*Status: overview. Written 2026-08-15, after E2 stage B.*

One pass through the evidence, in the order the argument runs. Every number is
read out of a committed artifact under `results/` — nothing here is retyped, so
this notebook goes stale loudly rather than quietly. The detail lives in the
four analysis notebooks and in `EXPERIMENT_LOG.md`; this is the read-first
version.

**The one-sentence claim.** Hidden-state geometry, measured as relative
Mahalanobis distance over a single forward pass, says *which problems a model
will fail* — it adds selective-prediction value on top of an eight-sample
self-consistency baseline on three models — and it does not say *which attempt
is right*. Most of what it knows is prompt difficulty.

**The shape of the paper.** A measurement-and-reframe paper, not a new
statistic. Half the contribution is a set of controls that shrink or kill prior
claims about reasoning geometry, this project's own included. The other half is
what is left standing after them.

**Two halves.** Sections 1–7 are the selective-prediction result and the controls
that bound it. Sections 8–13 are the causal thread — activation patching on a
synthetic arithmetic graph — which has no notebook of its own; this is the only
place it is written up outside `results/dag_patching/` and `EXPERIMENT_LOG.md`.
They share a question, not an experiment.

Details, in reading order: [12](12_wave1_abstention.ipynb) where geometry wins ·
[11](11_prompt_geometry_core_experiments.ipynb) where it does not ·
[13](13_deepconf_null_and_label_efficiency.ipynb) the baselines and the label
budget · [02](02_layer_dynamics.ipynb) the layer profile.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import fisher_exact

sys.path.insert(0, str(Path.cwd()))
import _viz_utils as vu

ROOT = vu.repo_root()

MODELS = {
    "qwen": "Qwen2.5-7B-Instruct",
    "deepseek": "DeepSeek-R1-Distill-Qwen-7B",
    "deepseek_llama": "DeepSeek-R1-Distill-Llama-8B",
}
BESTOFN = {label: f"results/{label}_bestofn_full/math500" for label in MODELS}
POPULATION = "cap_free_valid_plurality"


def load(path):
    with (ROOT / path).open(encoding="utf-8") as handle:
        return json.load(handle)


def ci(entry, digits=4):
    """point [low, high], the shape every bootstrap dict in these files uses."""
    return (
        f"{entry['point_estimate']:+.{digits}f} "
        f"[{entry['ci_low']:+.{digits}f}, {entry['ci_high']:+.{digits}f}]"
    )


INCREMENT = {
    label: load(f"{d}/math500_incremental_abstention_results.json")
    for label, d in BESTOFN.items()
}
# The solo-scorer stage never ran on the Llama distill, so this one dict is two
# models wide where the others are three.
BASELINES = {
    label: load(f"{d}/math500_abstention_baselines_results.json")
    for label, d in BESTOFN.items()
    if (ROOT / d / "math500_abstention_baselines_results.json").exists()
}
PEER = load("results/peer_difficulty_control/peer_difficulty_control_results.json")
ORGAD = load("results/orgad_agreement_control/orgad_agreement_control_results.json")
ALLOCATION = load("results/allocation_precheck/allocation_precheck_results.json")
DAG_POOLED = load("results/dag_patching/POOLED.json")
STAGE_B = load("results/dag_patching/e2_stage_b/ANALYSIS.json")["analysis"]

pd.set_option("display.width", 160)
print(f"repo root      {ROOT}")
print(f"models         {', '.join(INCREMENT)}  (solo scorers: {', '.join(BASELINES)})")
print(f"patching       pooled layer {DAG_POOLED['layer']}, stage B layer {STAGE_B['layer']}")

## 1. The object of study

Fit a reference manifold on hidden states from **correct** traces only — PCA to
128 dimensions, then a regularised Gaussian — and score a new trace by its
Mahalanobis distance to it, *minus* its distance to an unconditional background
Gaussian. That subtraction is the whole mechanism: raw Mahalanobis is close to
useless on reasoning traces, and going relative is worth +0.18 / +0.14 AUC.
`rmd_tail_q20` averages the per-token score over the final 20% of the trace.

The primitive is **not novel** — Vazhentsev et al. already define token-level
RMD and whole-trace ATRMD. The contribution is the evaluation.

| | |
|:--|:--|
| target | plurality-vote correctness over 8 sampled traces per prompt |
| `B0` | length, entropy, logprob, vote agreement — a self-consistency baseline |
| `B1` | `B0` + `rmd_tail_q20` |
| readout | out-of-fold logistic regression, prompt-clustered paired bootstrap, 1000 draws |
| metric | AURC, lower is better; **deltas only** across models, since levels inherit base accuracy |

Data: MATH-500, Best-of-8, three 7–8B models from two architecture families.
That is also the binding limitation — one dataset, one prompt set, two of the
three models reasoning-distilled and two Qwen-lineage.

## 2. The first control is the population

`collect_data.py` labels a trace with no parseable answer as incorrect, and
generation-capped traces dominate that class. The 2026-08-03 continuation study
says that label is wrong: capped traces resumed to 16,384 tokens **terminate 70%
[0.56, 0.81] of the time and are correct 46% of those**, against 5.6% as scored.
Capping is a budget shortfall, so those rows are censored observations, not
failures — which makes excluding them missing-data handling rather than dropping
inconvenient traces.

Everything headline below is on the cap-free population.

In [ ]:
rows = []
for label, payload in INCREMENT.items():
    for name in ("full_population", POPULATION):
        pop = payload["populations"][name]
        rows.append(
            {
                "model": label,
                "population": name,
                "prompts": pop["n_prompts"],
                "capped prompts": pop["n_capped_prompts"],
                "unparsed traces": pop["n_unparsed_traces"],
                "base accuracy": round(pop["base_accuracy"], 3),
            }
        )
print(pd.DataFrame(rows).to_string(index=False))

## 3. What did not survive the controls

Including several of this project's own earlier claims. This half of the paper
is the reason to trust the other half.

| Claim | Verdict |
|:--|:--|
| within-prompt geometry is "genuinely trace-level", AUC 0.93 | **retracted** — truncation artifact |
| distillation compresses correctness geometry to ~8 dimensions | **rejected** — parseable-only, DeepSeek climbs to 128 exactly like Qwen |
| the geometry effect is much stronger on DeepSeek than Qwen | **rejected** — it tracks 43% vs 8% truncation, not the model |
| the correct-reasoning manifold *shape* transfers across models | **weakened** — the readout half survives, the shape half in one direction on a weak signal |
| geometry rejects non-terminating traces | **do not say it** — length detects unparsed at 0.996 vs RMD 0.84; and it detects *unfinished*, not non-terminating |
| entropy-localised RMD is the method | **Qwen-specific** — untailed `rmd_full` recovers nearly the whole increment on both distilled models |
| one-class geometry is unusually sample-efficient | **narrowed** — see §7; it was measured against a probe differing in three ways at once |

Sources: `PAPER_STRATEGY.md` §3 and §7c, `EXPERIMENT_LOG.md` 2026-08-03 and
2026-08-09.

## 4. The headline: the increment

`B1 − B0` on the cap-free valid-plurality population. Negative favours geometry.

In [ ]:
rows = []
for label, payload in INCREMENT.items():
    pop = payload["populations"][POPULATION]
    delta = pop["paired_deltas"]["B1_minus_B0_aurc"]
    rows.append(
        {
            "model": MODELS[label],
            "n": pop["n_prompts"],
            "B0 AURC": round(pop["models"]["B0"]["metrics"]["aurc"], 4),
            "B1 AURC": round(pop["models"]["B1"]["metrics"]["aurc"], 4),
            "B1 - B0": ci(delta),
            "p": delta["p_two_sided"],
        }
    )
print(pd.DataFrame(rows).to_string(index=False))

fig, ax = plt.subplots(figsize=(7.5, 2.4))
order = list(reversed(list(MODELS)))
for i, label in enumerate(order):
    d = INCREMENT[label]["populations"][POPULATION]["paired_deltas"]["B1_minus_B0_aurc"]
    ax.plot([d["ci_low"], d["ci_high"]], [i, i], color="tab:blue", lw=2)
    ax.plot([d["point_estimate"]], [i], "o", color="tab:blue")
ax.axvline(0.0, color="grey", lw=1, ls="--")
ax.set_yticks(range(len(order)))
ax.set_yticklabels([MODELS[m] for m in order], fontsize=8)
ax.set_xlabel("B1 - B0 AURC  (negative favours geometry)")
ax.set_title("The increment over an eight-sample self-consistency baseline", fontsize=10)
fig.tight_layout()
plt.show()

## 5. Geometry on its own, against the cheap scorers

One scorer at a time, no readout: accuracy among the 50% of prompts it is most
confident about. This ran on two of the three models.

`length` is the baseline to watch — it is deceptively strong and prior
geometry/UQ work rarely benchmarks against it standalone.

In [ ]:
SOLO = ["rmd_tail_q20", "vote_agreement", "length", "logprob", "entropy",
        "conf_bottom10_group_ent"]
rows = []
for label, payload in BASELINES.items():
    point = payload["abstention"]["point"]
    row = {"model": label, "prompts": payload["abstention"]["n_prompts"]}
    row.update({m: round(point[m]["accuracy_at_coverage"]["0.5"], 3) for m in SOLO})
    rows.append(row)
solo = pd.DataFrame(rows)
print("accuracy at 50% coverage, single scorer, all 500 prompts")
print(solo.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 3.0))
width = 0.13
for i, method in enumerate(SOLO):
    ax.bar([j + i * width for j in range(len(solo))], solo[method], width=width, label=method)
ax.set_xticks([j + width * (len(SOLO) - 1) / 2 for j in range(len(solo))])
ax.set_xticklabels([MODELS[m] for m in solo["model"]], fontsize=8)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel("accuracy @ 50% coverage")
ax.legend(fontsize=7, ncol=3)
ax.set_title("One scorer at a time", fontsize=10)
fig.tight_layout()
plt.show()

## 6. Four controls the increment survives

**A. Length.** Partial trace length out of both scorers in rank space, then
compare against an uninformative scorer. Geometry keeps a large length-
independent component; on the reasoning-distilled model the *output-side*
baselines are the ones that turn out to be length proxies.

**B. DeepConf** (arXiv:2508.15260) as a prompt-level score, with all four of its
statistics. It is at chance on both models in all three framings, so there is no
external confidence baseline left to beat.

**C. The vote-proxy objection** (Orgad et al., arXiv:2410.02707): is geometry
just reading self-consistency? Inside the stratum where all eight siblings agree
and self-consistency is silent by construction, geometry still ranks.

**D. Difficulty.** Hand each model's `B0` the *other two models'* eight-sibling
pass rates — an empirical difficulty signal the target model did not produce.
This is the control that bites.

In [ ]:
print("A. length partialled out of both sides, vs an uninformative scorer (AUACC)")
rows = []
for label, payload in BASELINES.items():
    vs = payload["length_residualized"]["vs_uninformative"]
    rows.append(
        {
            "model": label,
            "rmd_tail_q20": ci(vs["rmd_tail_q20"]["auacc"], 3),
            "entropy": ci(vs["entropy"]["auacc"], 3),
            "logprob": ci(vs["logprob"]["auacc"], 3),
        }
    )
print(pd.DataFrame(rows).to_string(index=False))

print("\nB. geometry minus DeepConf's own statistic (accuracy @ 50% coverage)")
rows = []
for label, payload in BASELINES.items():
    deltas = payload["abstention"]["deltas"]
    rows.append(
        {
            "model": label,
            "vs bottom-10% group": ci(deltas["rmd_tail_q20_minus_conf_bottom10_group_ent"]["0.5"], 3),
            "vs tail entropy": ci(deltas["rmd_tail_q20_minus_conf_tail_q20_ent"]["0.5"], 3),
            "vs vote agreement": ci(deltas["rmd_tail_q20_minus_vote_agreement"]["0.5"], 3),
        }
    )
print(pd.DataFrame(rows).to_string(index=False))

print("\nC. inside the stratum where all eight siblings agree (AUROC)")
rows = []
for entry in ORGAD:
    unanimous = entry["strata"]["unanimous"]
    rows.append(
        {
            "model": entry["label"],
            "prompts": entry["n_prompts"],
            "unanimous": unanimous["n_prompts"],
            "share": round(unanimous["n_prompts"] / entry["n_prompts"], 2),
            "geometry AUROC there": ci(unanimous["auroc"], 3),
        }
    )
print(pd.DataFrame(rows).to_string(index=False))

print("\nD. peer-model pass rates inside the readout (AURC)")
rows = []
for entry in PEER["models"]:
    deltas = entry["populations"][POPULATION]["paired_deltas"]
    rows.append(
        {
            "model": entry["label"],
            "B1 - B0": ci(deltas["B1_minus_B0_aurc"]),
            "B1 - B0 | peer": ci(deltas["B1_minus_B0_given_peer_aurc"]),
            "peer - B0": ci(deltas["peer_minus_B0_aurc"]),
            "Holm p": PEER["holm"]["tests"][entry["label"]]["p_holm"],
        }
    )
print(pd.DataFrame(rows).to_string(index=False))
print("pre-declared stop rule triggered:", PEER["stop_rule"]["triggered"])

### Reading D honestly

The peer control is the first difficulty control here that beats `B0` on its own
— it cuts AURC by 28–82% where the two earlier ones were worth zero or less.
Against it the increment shrinks about **fivefold** and clears zero on two of
three models; Holm over the pre-declared family of three passes one.

DeepSeek-R1-Distill-Qwen-7B's null is a **ceiling, not redundancy**: with the
peer control its readout lands 0.0045 above a perfect ranker's AURC, leaving
nothing for any feature to remove.

So: **most of the increment is prompt difficulty.** That is a claim about the
mechanism, not a defeat — peer pass rates do not exist at decision time, so this
is a control the method is not required to beat. But the paper must say it in
the abstract, not the appendix.

## 7. The boundary — four things geometry does not do

- **Within-prompt correctness.** A pre-registered cross-model gate failed
  (2026-07-29): the localisation effect is +0.004 [−0.016, +0.027] on
  DeepSeek-R1-Distill-Qwen-7B against Qwen's +0.058, and *every* within-prompt
  AUC on that model — geometry and output baselines alike — sits at or below
  chance. Geometry indicates which problems are hard, not which attempt is right.
- **Best-of-N reranking.** Closed with a structural explanation: at N=8 only
  39/500 prompts tie and ~10 have headroom, a ~2-point ceiling; every tie-break
  delta is ≤0.006, p ≥ 0.248.
- **Label efficiency at scale.** The 50-label gap against a pooling-matched
  linear probe is −0.033 AURC, and the ladder splits it into −0.011 supervision
  + −0.018 decision-function *form*. Both rungs expire by 100 labels. The
  defensible sentence is that a positive-only fit is a cheap way to get a
  quadratic decision function, and the quadratic is where the advantage lives.
- **Sample allocation.** A pre-declared gate asked whether single-trace geometry
  predicts the gain from buying more samples, `g(p) = a(p,8) − a(p,1)`:

In [ ]:
print("target:", ALLOCATION["target"])
rows = []
for entry in ALLOCATION["models"]:
    across = entry["across_draws"]
    summary = entry["gain_summary"]
    rows.append(
        {
            "model": entry["label"],
            "share of prompts with zero gain": round(summary["share_exactly_zero"], 3),
            "rho(pass rate, gain)": round(summary["spearman_pass_rate_vs_gain"], 3),
            "geometry rank-corr with gain": round(across["spearman"]["geometry"]["median"], 3),
            "geometry R^2 on gain": round(across["r2_vs_constant"]["geometry"]["median"], 4),
            "geometry AUROC on difficulty": round(across["auroc_vs_prompt_outcome"]["median"], 3),
            "passes": ALLOCATION["gate"]["per_model"][entry["label"]]["passes"],
        }
    )
print(pd.DataFrame(rows).to_string(index=False))
print("\ngate:", ALLOCATION["gate"]["passes"], "|", ALLOCATION["gate"]["consequence"])

Geometry ranks the gain **backwards** on all three models while correlating
+0.51 / +0.24 / +0.37 with the pass rate. The one nominal pass sits at
R² = +0.0005 and is a coin flip; do not read 1 of 3 as partial support. The
target is also mostly zero — a prompt at 0/8 and one at 8/8 both gain nothing —
so nothing else predicts `g` either. Allocation is **closed**, not pending.
Routing and abstention by predicted difficulty are untouched, and the same
precheck supports them at one trace (AUROC 0.790 / 0.674 / 0.688).

## 8. The mechanism thread: what the intervention is

Everything above is a *readout* on states the model produced on its own. The
patching line asks a causal question instead, on a small synthetic arithmetic
family with a known computation graph (`DeepSeek-R1-Distill-Qwen-1.5B`,
`dag_patching.py`).

Each item is a short chain — `a = 7`, `m = a - 2`, … — ending in a question whose
answer is a single digit. Because the chain is affine and known, three different
digits can be named in advance for every item, and they are kept distinct by
construction:

- **target** — the clean answer;
- **implied** — the donor value *carried through the recipient's own chain*, what
  a propagated value would produce;
- **raw** — the digit literally standing at the patched position, what a readout
  that copies whatever it finds there would produce.

One forward pass writes the residual state at a chosen position into a second
pass, and the answer position's ten-digit distribution is read at four layers.
Every arm runs the first four row kinds below; the fifth needs a batch selected
for mutual donatability, so only the cross-item arms have it:

| row kind | what it edits | what it tests |
|:---|:---|:---|
| `ancestor` | a position the answer's graph depends on | the effect |
| `non_ancestor` | a matched position off the dependency path | is it the graph or the position |
| `null` | the same position, own state re-written | is the machinery inert |
| `surface_null` | a formatting rewrite, same token budget | is it the notation |
| `cross_item` | another item's state at the same position | are those positions just fragile |

Verdicts are three-valued and are a policy over the rows, not part of the
measurement: **invalid test** / **positive** / **scientific negative**. A rescore
never touches the artifacts. Two things to keep in view while reading the tables
below: the archived pilot was taken in bfloat16, whose 0.125-nat digit grid makes
exact two-digit ties ordinary, and every later run records its readout dtype.

In [ ]:
from collections import Counter

DAG = ROOT / "results/dag_patching"


def arm(relpath):
    with (DAG / relpath).open(encoding="utf-8") as handle:
        return json.load(handle)


# A file with rows is a patched arm; everything else in the tree is a manifest,
# a pool, a screening record or an analysis.
ARMS = {}
for path in sorted(DAG.rglob("*.json")):
    with path.open(encoding="utf-8") as handle:
        payload = json.load(handle)
    if isinstance(payload, dict) and "rows" in payload:
        ARMS[str(path.relative_to(DAG))] = payload

WHAT = {
    ".": "the archived pilot: immutable, provenance recovered after the fact",
    "paired_ladder": "ladder rerun on a family paired across depth",
    "v3_distinct": "ladder + cross-item, all three competing digits kept apart",
    "cross_item": "donor state taken from another item (earlier generator)",
    "written_vs_omitted": "does writing the intermediate value suppress the patch",
    "e2_stage_b": "the pre-registered matched-pair test",
}


def spread(values):
    # "unrecorded" is a finding here, not a gap to paper over.
    return ", ".join(sorted({"unrecorded" if v is None else str(v) for v in values}))


groups = {}
for name, payload in ARMS.items():
    groups.setdefault(str(Path(name).parent), []).append(payload)

rows = []
for family in ["."] + sorted(k for k in groups if k != "."):
    payloads = groups[family]
    verdicts = Counter(p["verdict"] for p in payloads)
    rows.append(
        {
            "family": family,
            "arms": len(payloads),
            "items": spread(p["n_items"] for p in payloads),
            "generator": spread(p.get("generator") for p in payloads),
            "readout": spread(p.get("readout_dtype") for p in payloads),
            "depth": spread(p.get("depth") for p in payloads),
            "verdicts": ", ".join(f"{k} x{n}" for k, n in sorted(verdicts.items())),
        }
    )
inventory = pd.DataFrame(rows)
print(f"{len(ARMS)} patched arms in {len(groups)} families")
print(inventory.to_string(index=False))
print()
for family in inventory["family"]:
    print(f"  {family:<20} {WHAT[family]}")

The three archived arms with an unrecorded depth are the v0 schema: `depth`,
`gap` and `ancestor_distance` were recovered by regenerating the items and
accepted only because the regenerated items reproduced the archived measurements
exactly. The unrecorded readout is the whole package — the dtype had to be traced
through the call chain afterwards, and `dag_pooling.pool` refuses to merge a
bfloat16 arm with a float32 one, so a change of instrument cannot enter a
numerator.

## 9. The depth ladder

The pooled counts below are `v3_distinct` only, at a **fixed layer 13**, over
items whose clean readout is uniquely correct, with exact-tie readouts held
apart from unique wins. Fixing one layer matters: at layer 20 the same
cross-item rows split 10 implied / 5 raw / 2 clean / 3 other, so a count taken at
each arm's own best layer would mix the effect with the layer choice.

The pool is offered every bfloat16 arm and keeps only the `v3_distinct` ones —
the two stage-B arms in §12 are float32, and `dag_pooling.pool` refuses to put a
change of instrument into a numerator.

In [ ]:
rows = []
for entry in DAG_POOLED["by_kind_and_depth"]:
    if entry["omit"] != "none":
        continue
    ok = entry["n_clean_correct_unique"]
    rows.append(
        {
            "donor": entry["kind"],
            "depth": entry["depth"],
            "seeds": len(entry["seeds"]),
            "items": entry["n_items"],
            "clean answer right": f"{ok}/{entry['n_items']}",
            "clean tied": entry["n_clean_tied"],
            "lands on implied": f"{entry['n_implied_top_unique']}/{ok}",
            "implied tied": entry["n_implied_tied"],
            "lands on raw": f"{entry['n_raw_top_unique']}/{ok}",
            "stays on clean": f"{entry['n_clean_top_unique']}/{ok}",
        }
    )
# The pool is offered every arm and keeps the ones it may keep: same generator,
# same readout precision. The refusals are the point, so count both.
contributing = {name for m in DAG_POOLED["measurements"] for name in m["arms"]}
print(
    f"generator {DAG_POOLED['generator']}, layer {DAG_POOLED['layer']}, "
    f"{DAG_POOLED['n_measurements']} measurements from {len(contributing)} of the "
    f"{DAG_POOLED['n_arms']} arms offered"
)
print(pd.DataFrame(rows).to_string(index=False))

print("\nsplit by clean confidence, same rows")
bands = [
    {
        "donor": e["kind"],
        "depth": e["depth"],
        "clean p(target)": f"{e['band'][0]:.1f}-{e['band'][1]:.1f}",
        "items": e["n_items"],
        "lands on implied": f"{e['n_implied_top_unique']}/{e['n_items']}",
        "implied tied": e["n_implied_tied"],
        "lands on raw": f"{e['n_raw_top_unique']}/{e['n_items']}",
    }
    for e in DAG_POOLED["by_confidence_band"]
    if e["omit"] == "none" and e["n_items"]
]
print(pd.DataFrame(bands).to_string(index=False))

In [ ]:
# gates.detail is one record per item per layer, so aggregate it here rather
# than quoting a stored summary.
LADDER = {depth: arm(f"v3_distinct/depth{depth}_gap0.json") for depth in (1, 2, 3)}
SERIES = [
    ("tv_ancestor", "ancestor"),
    ("tv_non_ancestor", "non-ancestor"),
    ("tv_surface_null", "surface null"),
    ("tv_null_max", "worst null"),
]

fig, axes = plt.subplots(1, 3, figsize=(10, 3.0), sharey=True)
for ax, (depth, payload) in zip(axes, LADDER.items()):
    detail = pd.DataFrame(payload["gates"]["detail"])
    median = detail.groupby("layer").median(numeric_only=True)
    for column, label in SERIES:
        ax.plot(median.index, median[column], marker="o", ms=4, label=label)
    ax.set_title(f"depth {depth}  ({payload['verdict']})", fontsize=9)
    ax.set_xlabel("layer")
    ax.set_xticks(payload["layer_bins"])
    # A shared axis is what shows the collapse, so print the small numbers.
    ax.annotate(
        f"ancestor at L13: {median.loc[13, 'tv_ancestor']:.3f}",
        xy=(0.5, 0.60), xycoords="axes fraction", ha="center", fontsize=7.5,
    )
axes[0].set_ylabel("median TV, clean to patched")
axes[-1].legend(fontsize=7, loc="upper right")
fig.suptitle("How far the edit moves the answer readout, by row kind", fontsize=10)
fig.tight_layout()
plt.show()

**A one-step channel, and nothing past it.** At depth 1 the ancestor patch
replaces the answer with the implied digit 21 of 23 times and never with the
donor's literal digit — and it does so while the non-ancestor, null and
surface-null rows leave the readout essentially where it was. At depth 2 and
depth 3 the answer stays put on all five items and the ancestor edit barely
disturbs the distribution at all — median TV 0.017 and 0.005 at layer 13 against
depth 1's 0.985, with `v3_distinct/README.md` recording that the clean answer
still holds 0.96 and 0.99 of the readout there. Those are **scientific
negatives**, not failed tests: the machinery ran, the controls behaved, the
effect was absent.

Three things this table cannot carry on its own, all recorded:

- **Layer 13 is a discovery layer.** It was chosen by looking at this very table,
  so the depth-1 rate here is not a held-out number. The right-hand panels also
  show why a *joint* layer had to be fixed: past layer 20 every row kind goes to
  zero, because there is no longer any downstream computation to disturb.
- **The `cross_item` row is exploratory.** Its donor batch is selected for mutual
  donatability, so it is not the ladder's value distribution, and an earlier
  cross-item arm on the previous generator failed its specificity leg outright.
- **Depth and clean confidence are collinear here.** Every depth-2 and depth-3
  item sits in the top confidence band while most depth-1 items sit below it —
  median clean p(target) 0.67 at depth 1 against 0.996 and 0.999 at depths 2 and
  3 (§10C, same generator and seed). In the archived runs the supports do not
  even touch: 0.666–0.961 against 0.966–0.999. So no item pair in this table
  separates depth from confidence. The band split above is inside depth 1 only,
  where the rate is flat (15/17 and 6/6). That confound is the whole reason for
  E2.

## 10. Three controls around the ladder

**Token distance.** If the effect were "these two positions are near the answer",
moving the patched line further away should weaken it. `depth1_gap{0,1,2}` place
the same spines at three distances.

**A donor from a different item.** The sharpest objection to the ancestor gap is
that those positions are simply perturbation-sensitive. The cross-item arm writes
*another item's* state there, same span and token width, under a derangement so
nothing donates to itself.

**A written intermediate result.** At depth 2 and 3 the trace states the chain's
values. `--omit chain` renders those lines without them, padded to the exact same
token count, so the graph and every downstream position are unchanged. `decoy`
omits as many values from lines the answer does *not* depend on.

In [ ]:
print("A. the same depth-1 spines at three token distances")
rows = []
for gap in (0, 1, 2):
    payload = arm(f"v3_distinct/depth1_gap{gap}.json")
    detail = pd.DataFrame(payload["gates"]["detail"])
    at13 = detail[detail["layer"] == 13]
    rows.append(
        {
            "gap": gap,
            "ancestor distance": f"{min(payload['ancestor_distance'])}-"
            f"{max(payload['ancestor_distance'])}",
            "median TV ancestor": round(at13["tv_ancestor"].median(), 4),
            "median TV worst null": round(at13["tv_null_max"].median(), 4),
            "answer moved gate": payload["gates"]["answer_moved"]["passes"],
            "verdict": payload["verdict"],
        }
    )
print(pd.DataFrame(rows).to_string(index=False))
print("these three arms share one spine set: 15 rows are 5 spines at 3 placements")

print("\nB. cross-item donor, four seeds")
rows = []
for seed in range(4):
    payload = arm(f"v3_distinct/cross_seed{seed}.json")
    gate = payload["gates"]["cross_item_donor"]
    rows.append(
        {
            "seed": seed,
            "verdict (within-item)": payload["verdict"],
            "cross-item gate measured": gate["measured"],
            "layers it clears": gate["layers"],
            "binds the verdict": gate["applied_to_verdict"],
        }
    )
print(pd.DataFrame(rows).to_string(index=False))

print("\nC. written versus omitted intermediate results, at layer 13")
rows = []
for name in [
    "depth1_none", "depth1_chain",
    "depth2_none", "depth2_decoy", "depth2_chain",
    "depth3_none", "depth3_decoy", "depth3_chain",
]:
    payload = arm(f"written_vs_omitted/{name}.json")
    spec = payload["gates"]["control_specificity"]["per_layer"]["13"]
    clean = payload["gates"]["clean_answer"]
    shares = [item["clean_probs"][item["target_value"]] for item in payload["items"]]
    rows.append(
        {
            "arm": name,
            "verdict": payload["verdict"],
            # bfloat16, so a tied clean readout is ordinary and is kept apart
            # from a unique win rather than folded into it.
            "clean top = target": f"{clean['n_unique_correct']}/{clean['n_items']}",
            "clean tied": clean["n_tied"],
            "median clean p(target)": round(pd.Series(shares).median(), 3),
            "ancestor -> implied": f"{spec['ancestor_implied']}/{spec['n_items']}",
            "ancestor moved": f"{spec['ancestor_moved']}/{spec['n_items']}",
            "controls moved": f"{spec['control_moved']}/{spec['n_control']}",
        }
    )
print(pd.DataFrame(rows).to_string(index=False))

**A. Distance is not the mechanism.** The ancestor edit replaces the answer at
every gap while the nulls stay flat, so the effect follows the dependency graph
rather than proximity to the read position. Its own limit is in the print: the
three arms are one spine set at three placements, not fifteen independent items.

**B. The cross-item gate is reported and never binds.** It is registered under
the joint-layer rule and sits beside the within-item verdict, because folding a
new statistic into a verdict before its null is known is exactly the post-hoc
move two earlier checkpoints were spent undoing. It clears at **no layer in three
of the four seeds** and at layer 6 only in the fourth — which is why the pooled
cross-item count in §9 (11/14 at layer 13) is an exploratory number and not a
control that passed. Read on mass rather than on the log-ratio it originally
reported, this arm's reading differs from the one the earlier generator gave.

**C. `depth2_chain`'s stored verdict is `positive` and should not be believed.**
Every gate in the arm scorer is relative, so an arm where the *background* moves
as much as the ancestor does clears all of them. The diagnostic beside the
verdict is what says otherwise: nulls flip the answer 23/40 and a comment rewrite
flips it 3/5, while the ancestor lands on its predicted digit 1/5. That is a
fragile readout, not a propagated value. The verdict is left as the scorer
produced it rather than patched by hand.

The `decoy` rows are what make the *clean-behaviour* ablation readable: same
notation, same token count, values omitted off the dependency path, and the model
is unchanged at 5/5 and p(target) 0.997. So the collapse in the `chain` arms is
not the model failing to read ` # # # #`. What the arms support is that **no
behaviourally usable carried intermediate was detected** — weaker than "there is
no latent computation", and deliberately so, since a behavioural failure after
removing a written value cannot separate computing it from retaining or
retrieving it.

## 11. E2 stage A — the confound, measured

Registered in `6f1e9a7` before the selection rule was written and before any of
these items existed. Stage A runs **clean forward passes only**; there is no
patched number in that directory and no code path in `dag_screening.py` that
could produce one, because it has to decide which items are comparable without
having seen how any of them respond to a patch.

The confound it targets: in the archived runs, depth-1 items are ones the model
was unsure of and depth-2 items ones it was sure of, and ancestor distance moves
with depth too. Screen wide enough at both depths and the supports overlap.

In [ ]:
SELECTION = load("results/dag_patching/e2_screening/SELECTION.json")
screened = pd.DataFrame(SELECTION["screened"])
sel = SELECTION["selection"]
low, high = sel["window"]

rows = []
for depth, frame in screened.groupby("depth"):
    eligible = frame[frame["clean_correct_unique"]]
    rows.append(
        {
            "depth": depth,
            "screened": len(frame),
            "eligible": len(eligible),
            "clean ties": int(frame["clean_tied"].sum()),
            "p(target) min": round(eligible["clean_target_share"].min(), 3),
            "median": round(eligible["clean_target_share"].median(), 3),
            "max": round(eligible["clean_target_share"].max(), 3),
            "distance min": int(eligible["ancestor_distance"].min()),
            "distance max": int(eligible["ancestor_distance"].max()),
        }
    )
print(f"readout {screened['readout_dtype'].unique()[0]}")
print(pd.DataFrame(rows).to_string(index=False))

pairs = sel["pairs"]
worst_p = max(abs(a["clean_target_share"] - b["clean_target_share"]) for a, b in pairs)
worst_d = max(abs(a["ancestor_distance"] - b["ancestor_distance"]) for a, b in pairs)
print(
    f"\nwindow [{low:.3f}, {high:.3f}] -> {sel['n_pairs']} pairs "
    f"(floor {sel['min_pairs']}, ceiling {sel['max_pairs']}), proceed = {sel['proceed']}"
)
print(f"worst pair: {worst_p:.4f} in clean p(target), {worst_d} token of ancestor distance")

fig, ax = plt.subplots(figsize=(7.5, 3.0))
for depth, frame in screened.groupby("depth"):
    eligible = frame[frame["clean_correct_unique"]]
    ax.hist(eligible["clean_target_share"], bins=40, alpha=0.55, label=f"depth {depth}")
ax.axvspan(low, high, color="grey", alpha=0.18, zorder=0)
ax.axvline(low, color="grey", lw=1)
ax.axvline(high, color="grey", lw=1)
ax.set_xlabel("clean p(target) — how sure the model already was")
ax.set_ylabel("eligible items")
ax.set_title("Stage A: where the two depths overlap at all (shaded = matching window)")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

Two readings worth keeping. **Not one tied clean readout in 1,230 items** in
float32, against 5 in 33 under bfloat16 — the ties were the recording precision,
not the model. And the overlap is real but thin and lives at the *top* of the
depth-1 range: the matched window is the high-confidence half of where the
original depth-1 result was obtained.

One hole in the rule, recorded rather than patched: it bounds ancestor distance
to ±2 tokens but bounds confidence only by ordering the greedy match. On a
smaller first screen that filled the ceiling with pairs 0.165 apart. Tripling the
screen removed the problem — the worst pair above is 0.0007 — so the rule was
left exactly as registered. On a small screen the hole is still there.

## 12. E2 stage B — the pre-registered test

Patch exactly those 24 pairs at layer 13. Nothing in the analysis moved between
the stages, and every item was re-checked against its stage-A measurement —
ancestor distance, target value, gap, and the clean readout to 1e-6 — before
being patched.

In [ ]:
primary = STAGE_B["primary"]
rows = []
for depth in ("1", "2"):
    validity = STAGE_B["validity"][depth]
    rows.append(
        {
            "depth": depth,
            "pairs": primary["n"][depth],
            "implied uniquely top": f"{primary['hits'][depth]}/{primary['n'][depth]}",
            "ancestor moved": f"{validity['ancestor']['flipped']}/{validity['ancestor']['n']}",
            "null rows flipped": f"{validity['null']['flipped']}/{validity['null']['n']}",
            "registered gate": "invalid" if validity["invalid_test"] else "valid",
            "arm scorer verdict": STAGE_B["verdict"][depth],
        }
    )
print(f"registered outcome: {primary['outcome']}, at layer {primary['layer']}")
print(pd.DataFrame(rows).to_string(index=False))

# The registration names one test. Fisher is the unpaired reading of the same
# 2x2 and is quoted beside the paired bootstrap, not instead of it.
fisher = fisher_exact(
    [
        [primary["hits"]["1"], primary["n"]["1"] - primary["hits"]["1"]],
        [primary["hits"]["2"], primary["n"]["2"] - primary["hits"]["2"]],
    ],
    alternative="greater",
)
print(
    f"\ndifference {primary['difference']:.2f}, interval "
    f"[{primary['interval'][0]:.3f}, {primary['interval'][1]:.3f}] over "
    f"{primary['replicates']} bootstrap replicates of whole pairs "
    f"(n = {primary['n_pairs']}); Fisher one-sided p = {fisher.pvalue:.1e}"
)
print("row kinds run:", ", ".join(STAGE_B["row_kinds"]),
      "| unreachable:", ", ".join(STAGE_B["unreachable_row_kinds"]))

In [ ]:
STAGE_B_ARMS = {depth: arm(f"e2_stage_b/depth{depth}.json") for depth in (1, 2)}

print("median TV at layer 13, clean to patched, by row kind")
rows = []
for depth, payload in STAGE_B_ARMS.items():
    frame = pd.DataFrame([r for r in payload["rows"] if r["layer"] == STAGE_B["layer"]])
    median = frame.groupby("kind")["tv"].median()
    rows.append({"depth": depth, **{k: round(v, 4) for k, v in median.items()}})
print(pd.DataFrame(rows).to_string(index=False))

print("\nlands on the implied digit, by layer (controls that moved, out of 192)")
rows = []
for depth in ("1", "2"):
    per_layer = STAGE_B["control_specificity"][depth]["per_layer"]
    row = {"depth": depth}
    for layer, entry in per_layer.items():
        row[f"L{layer}"] = (
            f"{entry['ancestor_implied']}/{entry['n_items']} ({entry['control_moved']})"
        )
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

levels = ["implied", "raw", "target", "other"]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.0), sharey=True)
for ax, depth in zip(axes, ("1", "2")):
    split = STAGE_B["level_split"][depth]
    x = range(len(levels))
    ax.bar([i - 0.2 for i in x], [split[k]["clean"] for k in levels], width=0.4, label="clean")
    ax.bar([i + 0.2 for i in x], [split[k]["patched"] for k in levels], width=0.4, label="patched")
    ax.set_xticks(list(x))
    ax.set_xticklabels([f"p({k})" for k in levels], fontsize=8)
    ax.set_title(f"depth {depth}  (n = {split['n']})", fontsize=10)
axes[0].set_ylabel("median probability")
axes[0].legend(fontsize=8)
fig.suptitle("Where the ancestor patch moves the digit readout, layer 13", fontsize=10)
fig.tight_layout()
plt.show()

**The contrast survives matching at full strength.** Clean-confidence quantiles
agree to three decimals and the distance supports overlap, and the rates did not
move toward each other at all: 24/24 against 0/24, against 0/192 control rows
that moved at any layer. The depth result is about graph depth, not about the
confidence or the token distance that travel with it.

Four things stated against that headline:

- **The interval is degenerate because the separation is perfect**, not because
  the estimate is precise. Zero discordant pairs leaves a resampler nothing to
  vary; read [1.000, 1.000] as "no item went the other way".
- **The depth-2 arm has two verdicts, recorded rather than resolved.** The
  registered gate — null flips ≥ 20% — calls it valid and negative (0/144). The
  project's own arm scorer calls it an *invalid test*, on
  `directional_control_failed` and `surface_above_null`, because its gates are
  relative and at depth 2 there is no movement for a relative gate to be relative
  to. Both labels are in the artifacts. They answer different questions, and the
  registration naming only the null-flip gate is a hole in the registration, not
  in the measurement.
- **The depth-2 patch is not inert, it is unaimed.** Ancestor TV is 0.088 against
  0.005 for a null — about twenty times a control — but the mass leaving the
  clean answer goes to the *other* digits (0.078 → 0.162) while p(implied) stays
  at 0.001.
- **At depth 1 the donor's literal digit is promoted ~200×** (0.0005 → 0.107)
  and still never wins (0/24). "The recipient transforms the donor value" stays
  too clean a sentence.

The layer table adds one thing the pooled ladder could not: the effect is 24/24
at layer 6 as well as at 13, 21/24 at 20, and gone at 27 — with controls at zero
throughout. Layer 13 is still inherited rather than re-searched, so this run is
confirmatory for the *contrast*, not for the layer.

## 13. What the patching line does and does not establish

**Does.** On this synthetic family, an ancestor-position residual patch installs
the value implied by the recipient's own chain at graph depth 1 and not at depth
2, and that contrast is not explained by clean confidence or token distance. The
controls are quiet wherever the clean readout is stable — non-ancestor, null and
surface-null rows leave the answer alone in every ladder arm and in both stage-B
arms. The exception is the two `chain` omission arms, where nulls flip the answer
23/40 and 33/40; those are exactly the arms whose clean behaviour had already
collapsed, which is why their patched numbers are not read as a result.

**Does not.** It says nothing about MATH-500, about the 7–8B models the rest of
the paper is measured on, or about natural traces — the family is synthetic and
the model is a 1.5B distill. It does not show *why* depth 2 fails: the mechanism
it is consistent with, the answer at depth 2 and 3 being fixed by the written
intermediate token that the patch does not touch, remains a hypothesis the
omission arms could not test. And it does not connect to `rmd_tail_q20`. The two
halves of the paper share a question, not an experiment.

**Untouched.** Depth 3 under matching; the cross-item row kind on matched items,
which a mutually-donatable batch cannot contain; the omission arms at any
precision above bfloat16; and any layer chosen on data that did not produce the
result.

## 14. Where this leaves the paper

**Standing claims.**

1. On MATH-500 after eight sampled traces, trace-mean relative Mahalanobis
   distance improves prompt-level selective prediction beyond length, entropy,
   log-probability, plurality agreement, and DeepConf, on three 7–8B models from
   two architecture families.
2. Most of that increment is prompt difficulty: peer-model pass rates absorb
   roughly four fifths of it, and one model's residual is a ceiling rather than
   a null.
3. The signal is **between-prompt**, not within-prompt — abstention and routing,
   not reranking or allocation.
4. The methodological half: length, generation caps, parse failures, layer
   selection, weak difficulty controls, and fixed-prediction uncertainty each
   inflate claims about reasoning geometry, demonstrated on claims this project
   itself made.
5. A separate, small, causal result: on a synthetic arithmetic family, an
   ancestor-position residual patch installs the implied answer at graph depth 1
   and not at depth 2, and that contrast survives a pre-registered match on
   clean confidence and token distance (24/24 vs 0/24).

**Open, in priority order.** Breadth is the binding constraint and it is now
*single-dataset* scope, not model count: OlympiadBench as a second prompt set,
and a second non-distilled model so "distilled vs not" stops being confounded
with "Qwen vs not". Both are queued and neither has run. On the mechanism side,
layer 13 needs re-searching on a family it was not chosen from, the cross-item
row kind is untested on matched items, and depth 3 and the omission arms are
untouched.

**How to falsify the headline.** Run the peer-difficulty control on a fourth
model and find the residual gone; or find a single dataset where `rmd_full` does
not clear length. Neither has been tried.